## Querying Lakebase Postgres via SDK

### Installing Utilities and Libraries

In [ ]:
%pip install psycopg[binary]==3.3.4 psycopg_pool==3.3.1 "databricks-sdk>=0.89.0"

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Setting up the Environment

In [ ]:
from databricks.sdk import WorkspaceClient
import psycopg

# Databricks SDK uses your existing OAuth identity
w = WorkspaceClient()

# Lakebase endpoint resource name
endpoint = (
    "projects//"
    "branches//"
    "endpoints/"
)

# Generate a short-lived OAuth credential
credential = w.postgres.generate_database_credential(
    endpoint=endpoint
)

In [ ]:
host = "LAKEBASE_HOTSNAME"
db_name = "LAKEBASE_DATABASE_NAME"
username = "LAKEBASE_USERNAME"
password = credential.token

### Create a Connection Pool

In [ ]:
from psycopg_pool import ConnectionPool

pool = ConnectionPool(
    conninfo=(
        f"host={host} "
        f"dbname={db_name} "
        f"user={username} "
        f"password={password} "
        f"sslmode=require"
    ),
    min_size=2,
    max_size=10
)

pool.wait()

print("Connection pool created successfully")

### Use the Connection Pool to Execute a SELECT Query

In [ ]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT *
            FROM ESG.Companies
        """)

        results = cur.fetchall()

        for row in results:
            print(row)
     

### Filter Data using WHERE

In [ ]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
        SELECT *
        FROM ESG.EmissionRecords
        WHERE Scope = %s
        """,
        (1,))

        results = cur.fetchall()

        for row in results:
            print(row)

### Filter Emissions greater than 500

In [ ]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
        SELECT *
        FROM ESG.EmissionRecords
        WHERE CO2_Emissions > %s
        """,
        (500,))

        results = cur.fetchall()

        for row in results:
            print(row)

### Sort by Emissions

In [ ]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT *
            FROM ESG.EmissionRecords
            ORDER BY CO2_Emissions ASC
        """)

        results = cur.fetchall()

        for row in results:
            print(row)

### Calculate Total Emissions per Company

In [ ]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                CompanyID,
                SUM(CO2_Emissions) AS TotalEmissions
            FROM ESG.EmissionRecords
            GROUP BY CompanyID
        """)

        results = cur.fetchall()

        for row in results:
            print(row)

### Get Pool Statistics 

In [ ]:
print(
    pool.get_stats()
)